# 02 - Data Preparation & EDA

Project: **RoadEye - Traffic Sign Recognition Using HOG Features and SVM** (Machine Learning Innovative Assignment).

> Run cells top to bottom. All numbers are computed when the cell runs; nothing is typed in by hand.

In [ ]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
from src import config, pipeline as P
np.random.seed(config.SEED)                      # reproducibility
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("RUN MODE:", config.RUN_MODE, "| seed:", config.SEED)

In [ ]:
def show(name, width=900):
    """Display a saved figure from results/figures/."""
    display(Image(filename=str(config.FIGURES_DIR / name), width=width))

def metrics(name):
    return json.loads((config.METRICS_DIR / name).read_text())

## 1. Prepare: clean, preprocess, split
`stage_prepare()`:
1. drops unreadable files (none expected),
2. ROI-crops, converts to grayscale, resizes to 32x32,
3. **splits by sign track** (`class_track` groups) so near-identical frames of one physical sign never appear in both train and validation,
4. keeps the official test set separate and untouched,
5. extracts HOG features (cached).

In [ ]:
data = P.stage_prepare()
info = metrics('split_info.json')
display(pd.Series({k: info[k] for k in ['run_mode','test_source','n_train','n_val','n_test','n_samples_after_cleaning','n_features_raw','n_features_hog','seed']}))
print('leakage checks:', info['leakage_checks'])

## 2. Class distribution per split (stratification check)

In [ ]:
cc = pd.DataFrame(info['class_counts']).fillna(0).astype(int)
cc['train_share'] = (cc['train'] / cc['train'].sum()).round(4)
cc['val_share'] = (cc['val'] / cc['val'].sum()).round(4)
display(cc.head(43))

## 3. Exploratory data analysis
Brightness distribution (illumination), most similar class pairs (by mean HOG vector) and outliers.

In [ ]:
eda = P.stage_eda(data)
display(pd.Series(eda['brightness']))
display(pd.read_csv(config.METRICS_DIR / 'eda_similar_classes.csv'))
display(pd.Series(eda['hog_feature_stats']))
show('eda_brightness_contrast.png', 800); show('eda_class_mean_images.png', 900); show('eda_similar_classes.png', 900); show('eda_size_by_class.png', 900); show('sample_images.png', 900)

## 4. Observations that drive decisions (fill in from the printed outputs)
* Imbalance ratio (max/min class count) -> decides the balancing strategy (`balancing.json`).
* Brightness spread / dark images -> HOG with `transform_sqrt=True` and block normalisation is more robust than raw pixels.
* Visually similar classes (speed limits, curve warnings) -> expect confusions there in the error analysis.

In [ ]:
bal = P.stage_balancing(data)
print(bal['decision'])
display(pd.DataFrame(bal['experiment_on_validation']) if bal['experiment_on_validation'] else pd.Series(bal['train_distribution']))